In [76]:
import anndata as ad
import decoupler as dc
import mofax as mfx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.stats.multitest import multipletests
import statsmodels.stats.multitest as multitest
from scipy.stats import mannwhitneyu

In [77]:
PATH_ADATA = "/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/adata/mofa_adata_30f.h5ad"
adata = ad.read_h5ad(PATH_ADATA)
views = adata.uns["mofa_views"]
factors = adata.uns["mofa_weights_factors"]
collectri = dc.get_collectri(organism="human")
tfs_of_interes = ["YAP1", "TEAD1", "TEAD2", "TEAD4", "WWTR1"]
results_per_view = {}

# Calcular actividad de TFs YAP/TAZ por linea celular usando los pesos de MOFA
for view in views:
    W = pd.DataFrame(
        adata.uns[f"mofa_weights_{view}"],
        index=adata.uns[f"mofa_weights_genes_{view}"],
        columns=factors
    )
    acts, pvals = dc.run_ulm(W.T, net=collectri, source="source", target="target", weight="weight")
    results_per_view[view] = {
        "acts": acts[tfs_of_interes],
        "pvals": pvals[tfs_of_interes]
    }

# Apilar resultados de todas las lineas celulares
acts_all = pd.concat({view: res["acts"] for view, res in results_per_view.items()}, names=["cell_line", "factor"]).reset_index()
pvals_all = pd.concat({view: res["pvals"] for view, res in results_per_view.items()}, names=["cell_line", "factor"]).reset_index()

acts_long = acts_all.melt(id_vars=["cell_line", "factor"], var_name="TF", value_name="actividad")
pvals_long = pvals_all.melt(id_vars=["cell_line", "factor"], var_name="TF", value_name="pval")

# FDR global sobre todos los p-valores juntos (mas conservador)
pvals_long["padj"] = multipletests(pvals_long["pval"], method="fdr_bh")[1]

resultados_df = acts_long.copy()
resultados_df["padj"] = pvals_long["padj"].values

# Factores con actividad negativa y significativa
factores_interes = resultados_df[
    (resultados_df["padj"] < 0.05) &
    (resultados_df["actividad"] < 0)
].reset_index(drop=True)

print(f"Combinaciones significativas y negativas: {factores_interes.shape}")
print(factores_interes)

Combinaciones significativas y negativas: (48, 5)
     cell_line    factor     TF  actividad      padj
0       BT-474  Factor29   YAP1  -3.711217  0.028503
1          C32   Factor5   YAP1  -3.523778  0.042347
2    HEPG2_C3A  Factor25   YAP1  -3.795392  0.025429
3    NCI-H1573  Factor25   YAP1  -3.506178  0.043680
4     NCI-H460  Factor12   YAP1  -4.064977  0.013049
5       SW 900  Factor12   YAP1  -3.503598  0.043680
6       SW1417  Factor25   YAP1  -3.688569  0.030085
7        SW480   Factor5   YAP1  -4.895458  0.001598
8        A-172   Factor5  TEAD1  -4.270532  0.008635
9         A498   Factor5  TEAD1  -3.738959  0.027222
10        A549   Factor5  TEAD1  -3.533802  0.041356
11      AN3 CA   Factor5  TEAD1  -4.573961  0.003774
12      C-33 A   Factor5  TEAD1  -4.061390  0.013049
13         C32   Factor5  TEAD1  -3.547337  0.040636
14     CFPAC-1   Factor5  TEAD1  -3.980644  0.016140
15     CHP-212   Factor8  TEAD1  -3.893004  0.020499
16    COLO 205   Factor5  TEAD1  -6.379978  0.000

AQUI ES DONDE EMPIEZO A UTILIZAR LOS DATOS DE DEPMAP 

In [78]:
import requests
import os
prism_article_id = '25917643'
crispr_article_id = '25880521'
ruta_salida = "/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/"
for article_id, nombres in [
    (prism_article_id, ['Repurposing_Public_24Q2_LFC.csv',
                        'Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv',
                        'Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv',
                        'Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv']),
    (crispr_article_id, ['CRISPRGeneEffect.csv'])
]:
    response = requests.get(f'https://api.figshare.com/v2/articles/{article_id}/files')
    files = response.json()
    
    for f in files:
        if f['name'] in nombres:
            print(f"Descargando {f['name']}...")
            r = requests.get(f['download_url'])
            with open(os.path.join(ruta_salida, f['name']), 'wb') as out:
                out.write(r.content)
            print(f"Guardado: {f['name']}")


Descargando Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv...
Guardado: Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv
Descargando Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv...
Guardado: Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv
Descargando Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv...
Guardado: Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv
Descargando Repurposing_Public_24Q2_LFC.csv...
Guardado: Repurposing_Public_24Q2_LFC.csv
Descargando CRISPRGeneEffect.csv...
Guardado: CRISPRGeneEffect.csv


In [79]:
cell_line_metadata = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv")
metadata = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv")
epdm = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv")
crispr = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/CRISPRGeneEffect.csv", index_col=0)
print(f"CRISPR: {crispr.shape}, DepMap: {epdm.shape}")

CRISPR: (1150, 18443), DepMap: (6790, 920)


In [80]:
# Definir líneas celulares YAP/TAZ-dependientes usando CRISPR gene effect scores
# Umbral < -0.5: líneas que necesitan ese gen para sobrevivir
yap_col = [col for col in crispr.columns if col.startswith("YAP1")][0]
wwtr1_col = [col for col in crispr.columns if col.startswith("WWTR1")][0]
tead_cols = [col for col in crispr.columns if any(col.startswith(t) for t in ["TEAD1", "TEAD2", "TEAD4"])]
# Pongo el limite en el -0.5 ya que me indica que silenciar ese gen reduce la supervivencia de la linea celular, lo que sugiere dependencia. Es un umbral comúnmente usado en análisis de dependencia genética.
yap_dependent = crispr[
    (crispr[yap_col] < -0.5) |
    (crispr[wwtr1_col] < -0.5) |
    crispr[tead_cols].lt(-0.5).any(axis=1)
].index.tolist()

print(f"Líneas YAP/TAZ-dependientes: {len(yap_dependent)}")

Líneas YAP/TAZ-dependientes: 560


In [81]:
# Preparar tabla de DepMap en formato largo: depmap_id x BRD_ID → LFC
epm = epdm.rename(columns={'Unnamed: 0': 'BRD_ID'}).set_index('BRD_ID')
rep_clean = metadata[['IDs', 'Drug.Name', 'MOA', 'Synonyms']].drop_duplicates('IDs').rename(columns={'IDs': 'BRD_ID'})

epm_t = epm.T
epm_t.index.name = 'depmap_id'
epm_t = epm_t.reset_index().merge(cell_line_metadata[['depmap_id', 'ccle_name']], on='depmap_id', how='left')
epm_t['cell_line'] = epm_t['ccle_name'].str.split('_').str[0]

# Verificar solapamiento con líneas celulares de MOFA
comunes = set(epm_t['cell_line'].str.upper().str.replace(r'[-\s]', '', regex=True)) & \
          set(pd.Series(adata.uns['mofa_views']).str.upper().str.replace(r'[-\s]', '', regex=True))
print(f"Líneas celulares en común con MOFA: {len(comunes)}")


Líneas celulares en común con MOFA: 40


In [82]:
# Pasar a formato largo excluyendo columnas de metadata
brd_cols = [c for c in epm_t.columns if c not in ['depmap_id', 'ccle_name', 'cell_line', 'index']]
epm_long = epm_t.melt(id_vars=['depmap_id', 'cell_line'], value_vars=brd_cols, var_name='BRD_ID', value_name='LFC').dropna(subset=['LFC'])
epm_long = epm_long.merge(rep_clean[['BRD_ID', 'Drug.Name']], on='BRD_ID').drop_duplicates()
print(epm_long.shape)


(4218691, 5)


In [83]:
# tabla_final y merged: se usaban para ver las lineas celulares comunes entre MOFA y DepMap
# y para extraer los factores unicos. No son necesarias para el analisis principal
# ya que los factores se obtienen directamente de factores_interes['factor'].unique()
# tabla_final = epm_long.merge(factores_interes, on='cell_line', how='inner').dropna(subset=['LFC'])
# drogas_mofa = set(adata.obs['drug'].unique())
# merged = tabla_final[tabla_final['Drug.Name'].isin(drogas_mofa)]
# merged['cell_line'].unique()

In [84]:
# Análisis principal: ¿las drogas que inhiben YAP/TAZ según MOFA matan más
# las células YAP-dependientes en DepMap que las no dependientes?

# Scores de MOFA por droga (media entre concentraciones y placas)
scores = pd.DataFrame(adata.X, index=adata.obs_names, columns=adata.var_names)
scores['drug'] = adata.obs['drug']
scores_droga = scores.groupby('drug', observed=True).mean()

# Top 20 drogas más negativas por factor
top_drogas_por_factor = {
    factor: scores_droga[factor].nsmallest(10).index.tolist()
    for factor in factores_interes['factor'].unique()
}

resultados = []
for factor, drogas in top_drogas_por_factor.items():
    brd_ids = rep_clean[rep_clean['Drug.Name'].isin(drogas)]['BRD_ID'].tolist()

    lfc_yap = epm_long[
        (epm_long['BRD_ID'].isin(brd_ids)) &
        (epm_long['depmap_id'].isin(yap_dependent))
    ]['LFC'].dropna()

    lfc_no_yap = epm_long[
        (epm_long['BRD_ID'].isin(brd_ids)) &
        (~epm_long['depmap_id'].isin(yap_dependent))
    ]['LFC'].dropna()

    if len(lfc_yap) < 5 or len(lfc_no_yap) < 5:
        continue

    stat, pval = mannwhitneyu(lfc_yap, lfc_no_yap, alternative='less')

    resultados.append({
        'factor': factor,
        'lfc_medio_yap': lfc_yap.mean(),
        'lfc_medio_no_yap': lfc_no_yap.mean(),
        'pval': pval,
        'n_drogas': len(drogas)
    })

resultados_df = pd.DataFrame(resultados)
mask = resultados_df['pval'].notna()
resultados_df.loc[mask, 'padj'] = multipletests(resultados_df.loc[mask, 'pval'], method='fdr_bh')[1]
print(resultados_df.sort_values('pval'))


     factor  lfc_medio_yap  lfc_medio_no_yap          pval  n_drogas      padj
5   Factor6      -2.143621         -1.889880  2.390852e-07        10  0.000002
4   Factor8      -1.334918         -1.156345  4.629833e-03        10  0.018079
0  Factor29      -1.659547         -1.535266  6.779533e-03        10  0.018079
2  Factor25      -1.831902         -1.725964  1.747879e-02        10  0.034958
3  Factor12      -1.484627         -1.398011  1.035825e-01        10  0.165732
7  Factor18      -1.310898         -1.274707  2.904433e-01        10  0.332460
1   Factor5      -1.680842         -1.648417  2.909028e-01        10  0.332460
6   Factor3      -1.195454         -1.171023  7.751351e-01        10  0.775135


In [87]:
# Para los factores significativos
factores_sig = ['Factor6', 'Factor8','Factor29','Factor25']

rows = []
for factor in factores_sig:
    drogas = top_drogas_por_factor[factor]
    brd_ids = rep_clean[rep_clean['Drug.Name'].isin(drogas)]['BRD_ID'].tolist()
    
    for drug in drogas:
        brd = rep_clean[rep_clean['Drug.Name'] == drug]['BRD_ID'].tolist()
        lfc_yap = epm_long[(epm_long['BRD_ID'].isin(brd)) & (epm_long['depmap_id'].isin(yap_dependent))]['LFC'].mean()
        lfc_no_yap = epm_long[(epm_long['BRD_ID'].isin(brd)) & (~epm_long['depmap_id'].isin(yap_dependent))]['LFC'].mean()
        rows.append({'factor': factor, 'drug': drug, 'lfc_yap': lfc_yap, 'lfc_no_yap': lfc_no_yap, 'diff': lfc_yap - lfc_no_yap})

drug_df = pd.DataFrame(rows).sort_values(['factor', 'diff'])
print(drug_df)

      factor                          drug   lfc_yap  lfc_no_yap      diff
31  Factor25                   VINCRISTINE -1.989633   -1.823046 -0.166587
37  Factor25                     AURANOFIN -5.761637   -5.632939 -0.128698
35  Factor25                HYDROXYFASUDIL -0.470174   -0.376616 -0.093558
34  Factor25                     PONATINIB -2.150945   -2.063558 -0.087387
38  Factor25                      IXAZOMIB -4.312079   -4.251277 -0.060802
32  Factor25                    PERETINOIN -0.235020   -0.208049 -0.026971
36  Factor25                   TOFACITINIB -0.049760   -0.031502 -0.018258
30  Factor25                    DAPTOMYCIN -0.014299   -0.001563 -0.012735
33  Factor25         VINBLASTINE_(SULFATE)       NaN         NaN       NaN
39  Factor25                   IPATASERTIB       NaN         NaN       NaN
24  Factor29                   VINCRISTINE -1.989633   -1.823046 -0.166587
21  Factor29                HYDROXYFASUDIL -0.470174   -0.376616 -0.093558
22  Factor29             

In [88]:
# Guardar resultados en Excel
output_path = "/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/yap_taz_results.xlsx"

# Tabla combinada: score MOFA + LFC DepMap por droga y factor
top_drogas_df = pd.DataFrame([
    {"factor": factor, "rank_mofa": i+1, "drug": drug,
     "mofa_score": scores_droga.loc[drug, factor] if drug in scores_droga.index else None}
    for factor, drogas in top_drogas_por_factor.items()
    for i, drug in enumerate(drogas)
])

# Merge con LFC de DepMap (drug_df calculado en celda anterior)
tabla_combinada = top_drogas_df.merge(
    drug_df[["factor", "drug", "lfc_yap", "lfc_no_yap", "diff"]],
    on=["factor", "drug"], how="left"
).sort_values(["factor", "rank_mofa"])

with pd.ExcelWriter(output_path) as writer:
    resultados_df.sort_values("pval").to_excel(writer, sheet_name="mannwhitney_por_factor", index=False)
    tabla_combinada.to_excel(writer, sheet_name="top10_drogas_por_factor", index=False)
    factores_interes.to_excel(writer, sheet_name="factores_yap_significativos", index=False)

print(f"Guardado en {output_path}")
print(tabla_combinada[tabla_combinada["factor"] == "Factor6"])

Guardado en /mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/yap_taz_results.xlsx
     factor  rank_mofa                   drug  mofa_score   lfc_yap  \
50  Factor6          1             DINACICLIB   -2.635611 -4.061371   
51  Factor6          2      HOMOHARRINGTONINE   -1.845398 -4.160148   
52  Factor6          3            SBI-0640756   -1.217833       NaN   
53  Factor6          4          HARRINGTONINE   -0.895583 -3.251489   
54  Factor6          5             BELZUTIFAN   -0.843684  0.061721   
55  Factor6          6                TAK-901   -0.719797 -2.007869   
56  Factor6          7            PEMIGATINIB   -0.667928 -0.940897   
57  Factor6          8  OUABAIN_(OCTAHYDRATE)   -0.645397       NaN   
58  Factor6          9              DIGITOXIN   -0.613340 -3.364602   
59  Factor6         10         HYDROXYFASUDIL   -0.472491 -0.470174   

    lfc_no_yap      diff  
50   -3.924066 -0.137305  
51   -4.112849 -0.047299  
52         NaN       NaN  
53   -3.183037 -0.068452  
5